# PyTorch 문법과 Dense Layer 이해하기

이 노트북은 `mynote_lecture1.ipynb`에 작성한 `MyDenseLayer`를 이해하기 위해 필요한 PyTorch 문법을 순서대로 설명합니다. 각 코드 셀은 위에서 아래로 실행하면 됩니다.

학습 목표:

1. Tensor와 shape 이해하기
2. 행렬 곱과 broadcasting 이해하기
3. `nn.Module`, `__init__`, `forward` 이해하기
4. `nn.Parameter`와 자동 미분 이해하기
5. 사용자 정의 Dense Layer를 만들고 학습하기

## 1. PyTorch 불러오기

`torch`는 텐서와 수학 연산을 제공하고, `torch.nn`은 신경망을 구성하는 클래스와 함수를 제공합니다. 관례적으로 `torch.nn`은 `nn`이라는 별칭으로 사용합니다.

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

## 2. Tensor와 shape

Tensor는 PyTorch가 사용하는 다차원 숫자 배열입니다. 신경망에서 한 행은 보통 하나의 데이터 샘플이고, 한 열은 하나의 특징(feature)을 의미합니다.

아래 `x`에는 샘플 2개가 있고, 각 샘플은 특징 3개를 가집니다. 따라서 shape은 `(2, 3)`입니다.

In [ ]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print(x)
print("shape:", x.shape)
print("dtype:", x.dtype)
print("차원 수:", x.ndim)

### 자주 사용하는 Tensor 생성 함수

- `torch.zeros(a, b)`: 모든 원소가 0인 Tensor
- `torch.ones(a, b)`: 모든 원소가 1인 Tensor
- `torch.rand(a, b)`: 0 이상 1 미만의 균등분포 난수
- `torch.randn(a, b)`: 평균 0, 표준편차 1인 정규분포 난수

`torch.manual_seed()`를 사용하면 같은 난수를 다시 만들 수 있어 실험을 재현하기 좋습니다.

In [ ]:
torch.manual_seed(42)

print("zeros:\n", torch.zeros(2, 3))
print("ones:\n", torch.ones(2, 3))
print("uniform random:\n", torch.rand(2, 3))
print("normal random:\n", torch.randn(2, 3))

## 3. 행렬 곱과 Broadcasting

Dense Layer의 핵심 계산은 다음과 같습니다.

$$Z = XW + b$$

`X @ W`에서 X의 마지막 차원과 W의 첫 번째 차원이 같아야 합니다.

- `X`: `(batch_size, input_dim)`
- `W`: `(input_dim, output_dim)`
- `X @ W`: `(batch_size, output_dim)`

편향 `b`의 shape이 `(1, output_dim)`이면 PyTorch가 각 샘플에 같은 편향을 자동으로 더합니다. 이를 broadcasting이라고 합니다.

In [ ]:
torch.manual_seed(42)

X = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])                         # shape: (2, 3)
W = torch.randn(3, 2)      # shape: (3, 2)
b = torch.randn(1, 2)      # shape: (1, 2)

Z = X @ W + b              # torch.matmul(X, W)와 동일

print("X shape:", X.shape)
print("W shape:", W.shape)
print("b shape:", b.shape)
print("Z shape:", Z.shape)
print("Z:\n", Z)

## 4. `nn.Module`로 레이어 만들기

사용자 정의 신경망과 레이어는 `nn.Module`을 상속합니다.

- `__init__`: 레이어와 학습할 파라미터를 생성합니다.
- `super().__init__()`: 부모인 `nn.Module`을 초기화합니다. 이 호출이 있어야 PyTorch가 파라미터와 하위 레이어를 관리할 수 있습니다.
- `forward`: 입력이 들어왔을 때 수행할 순전파 연산을 정의합니다.
- `layer(x)`: `forward`를 직접 부르기보다 객체를 함수처럼 호출합니다. PyTorch가 내부적으로 `forward(x)`를 실행합니다.

In [ ]:
class MyDenseLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        # nn.Parameter로 감싸면 모델이 학습해야 할 값으로 등록됩니다.
        # nn.Parameter는 기본적으로 requires_grad=True입니다.
        self.W = nn.Parameter(torch.randn(input_dim, output_dim))
        self.b = nn.Parameter(torch.randn(1, output_dim))

    def forward(self, inputs):
        # 1. 선형 결합: Z = XW + b
        z = torch.matmul(inputs, self.W) + self.b

        # 2. 비선형 활성화 함수 적용
        output = torch.sigmoid(z)
        return output

### 사용자 정의 레이어 실행하기

입력 차원이 3이고 출력 뉴런이 2개인 레이어를 생성합니다. 입력 X의 shape이 `(2, 3)`이면 출력은 `(2, 2)`가 됩니다. Sigmoid를 통과하므로 모든 출력값은 0과 1 사이입니다.

In [ ]:
torch.manual_seed(42)

layer = MyDenseLayer(input_dim=3, output_dim=2)
output = layer(X)

print("output:\n", output)
print("output shape:", output.shape)

print("\n등록된 학습 파라미터:")
for name, parameter in layer.named_parameters():
    print(name, parameter.shape, "requires_grad=", parameter.requires_grad)

## 5. 자동 미분과 역전파

PyTorch는 `requires_grad=True`인 Tensor가 참여한 연산을 기록합니다. 최종 스칼라 값에서 `backward()`를 호출하면 연쇄법칙으로 미분을 계산하고 각 파라미터의 `.grad`에 저장합니다.

아래 예제에서 $y=x^2$이므로 $dy/dx=2x$입니다. x가 2이면 gradient는 4입니다.

In [ ]:
x_scalar = torch.tensor(2.0, requires_grad=True)
y_scalar = x_scalar ** 2
y_scalar.backward()

print("y =", y_scalar.item())
print("dy/dx =", x_scalar.grad.item())

## 6. 실무에서는 `nn.Linear` 사용하기

직접 W와 b를 만들지 않고 `nn.Linear(input_dim, output_dim)`을 사용하는 것이 일반적입니다. `nn.Linear`는 파라미터 초기화와 선형 결합을 처리합니다.

주의: 직접 구현한 W의 shape은 `(input_dim, output_dim)`이지만, `nn.Linear.weight`의 shape은 `(output_dim, input_dim)`입니다. `nn.Linear`가 내부적으로 $XW^T+b$를 계산하므로 결과 shape은 동일합니다.

In [ ]:
simple_layer = nn.Sequential(
    nn.Linear(in_features=3, out_features=2),
    nn.Sigmoid()
)

simple_output = simple_layer(X)

print(simple_layer)
print("output shape:", simple_output.shape)
print("Linear weight shape:", simple_layer[0].weight.shape)
print("Linear bias shape:", simple_layer[0].bias.shape)

## 7. 작은 모델 학습하기

학습의 기본 순서는 다음과 같습니다.

1. `prediction = model(x)`: 순전파
2. `loss = loss_fn(prediction, target)`: 손실 계산
3. `optimizer.zero_grad()`: 이전 gradient 초기화
4. `loss.backward()`: 역전파로 gradient 계산
5. `optimizer.step()`: 파라미터 업데이트

PyTorch는 gradient를 기본적으로 누적하므로 매 반복에서 `zero_grad()`가 필요합니다.

In [ ]:
torch.manual_seed(42)

train_x = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])
train_y = torch.tensor([[0.0], [0.0], [0.0], [1.0]])  # AND 문제

model = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(1000):
    prediction = model(train_x)
    loss = loss_fn(prediction, train_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.6f}")

## 8. 평가하기

평가할 때는 `model.eval()`로 평가 모드를 설정하고 `torch.inference_mode()`로 gradient 기록을 끕니다. 현재 모델에는 Dropout이나 Batch Normalization이 없어서 `eval()`에 따른 수치 변화는 없지만, 실무에서 사용하는 습관을 익히는 것이 좋습니다.

In [ ]:
model.eval()

with torch.inference_mode():
    probabilities = model(train_x)
    predictions = (probabilities >= 0.5).float()

print("확률:\n", probabilities)
print("예측 클래스:\n", predictions)
print("정답:\n", train_y)

## 9. CPU와 Apple Silicon GPU(MPS)

Apple Silicon Mac에서는 NVIDIA CUDA 대신 MPS를 사용할 수 있습니다. 모델과 입력 데이터는 반드시 같은 device에 있어야 합니다. 작은 예제는 CPU로도 충분합니다.

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("사용 가능한 device:", device)

# 실제 이동 방법
# model = model.to(device)
# train_x = train_x.to(device)
# train_y = train_y.to(device)

## 핵심 정리

- PyTorch의 기본 데이터 단위는 `Tensor`입니다.
- 신경망 클래스는 `nn.Module`을 상속합니다.
- 레이어와 파라미터는 `__init__`에서 만들고 계산 과정은 `forward`에 작성합니다.
- `nn.Parameter`나 `nn.Linear`의 파라미터는 자동으로 모델에 등록됩니다.
- `loss.backward()`가 gradient를 계산하고 `optimizer.step()`이 파라미터를 업데이트합니다.
- Dense Layer의 핵심 계산은 $Y=\sigma(XW+b)$입니다.
- 실무에서는 직접 행렬을 만드는 대신 보통 `nn.Linear`를 사용합니다.